# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shakir-j/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content Lifecycle: Growing vs Declining

The paper reports that growing pages were younger on average than declining pages: 185 days versus 228 days. The paper also notes that word count was nearly the same between the two groups. This is an observed difference in the dataset.

**Source:** FlyRank, *The State of AI-Driven SEO* (April 2026).

**My methodology question:** Where exactly does the growing/declining label come from, and is it defined using a future performance window that is separate from the age and other variables being compared? I would also want to know whether the comparison controls for differences between clients or sites, because client mix could contribute to the observed age difference.

This is a constructive question because the finding is useful as an observed pattern, but the validation design should make clear whether the comparison supports a broader claim beyond this dataset.

### Finding 2 — The CTR Cliff

The paper reports a strong relationship between search position and click-through rate, describing a sharp decline in CTR as pages move further down the search results. The analysis uses search visibility data and compares CTR across position groups.

**Source:** FlyRank, *The State of AI-Driven SEO* (April 2026).

**My methodology question:** Does the validation design support interpreting this relationship as a general directional pattern rather than evidence that position itself causes the CTR change? I would want to know how the position groups are constructed, whether CTR is weighted consistently across groups, and whether differences in query intent, brand effects, or other page characteristics could contribute to the observed relationship.

I would therefore describe this as a measured association in the study data rather than a causal effect.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before: Random row-level split

The random row-level split measured accuracy of 0.8179, precision of 0.8944, recall of 0.4838, and F1 of 0.6280. However, 53 clients appeared in both the training and test sets. This means the test set was not independent at the client level.

### After: Client-grouped split

The client-grouped split measured accuracy of 0.7837, precision of 0.8896, recall of 0.4758, and F1 of 0.6200. There were 43 training clients and 11 test clients, with 0 client overlap.

The grouped evaluation is more appropriate for assessing generalization to unseen clients. Compared with the random split, the measured accuracy decreased from 0.8179 to 0.7837 and F1 decreased from 0.6280 to 0.6200. Precision and recall also decreased slightly.

I therefore treat the client-grouped results as the more honest evaluation for this question. The difference does not prove that the random split was biased, but the client overlap means its results could be more optimistic about performance on unseen clients.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 2 — Before/after validation audit

# ============================================================
# SECTION 2 — HONEST SPLIT: BEFORE / AFTER
# ============================================================

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ------------------------------------------------------------
# 1. FlyRank warehouse setup
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise RuntimeError(
        "HF_TOKEN not found. Check Colab Secrets and make sure "
        "the secret is named exactly HF_TOKEN."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"


# ------------------------------------------------------------
# 2. Recreate the exact Week-5 feature frame
# ------------------------------------------------------------

feature_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(COALESCE(gsc_impressions, 0))
            AS feb_gsc_impressions,

        SUM(COALESCE(gsc_clicks, 0))
            AS feb_gsc_clicks,

        AVG(gsc_avg_position)
            AS feb_gsc_avg_position,

        SUM(COALESCE(ga4_sessions, 0))
            AS feb_ga4_sessions,

        SUM(COALESCE(scroll_events, 0))
            AS feb_scroll_events

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        CASE
            WHEN
                SUM(COALESCE(gsc_clicks, 0)) > 0
                OR SUM(COALESCE(ga4_sessions, 0)) > 0
                OR SUM(COALESCE(scroll_events, 0)) > 0
            THEN 1
            ELSE 0
        END AS march_activity_label

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,

    f.feb_gsc_impressions,
    f.feb_gsc_clicks,
    f.feb_gsc_avg_position,
    f.feb_ga4_sessions,
    f.feb_scroll_events,

    COALESCE(m.march_activity_label, 0)
        AS march_activity_label

FROM feb f

LEFT JOIN mar m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id
""").df()


# ------------------------------------------------------------
# 3. Exact Week-5 feature set and target
# ------------------------------------------------------------

feature_cols = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_ga4_sessions",
    "feb_scroll_events"
]

X = feature_frame[feature_cols].copy()
y = feature_frame["march_activity_label"].astype(int)
groups = feature_frame["client_hash_id"]


print("Feature frame shape:", feature_frame.shape)
print("Feature columns:", feature_cols)
print("Positive label rate:", round(y.mean(), 4))


# ------------------------------------------------------------
# 4. Same Logistic Regression used in Week 5
# ------------------------------------------------------------

def build_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )


def evaluate_model(X_train, X_test, y_train, y_test):
    model = build_model()

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        )
    }


# ------------------------------------------------------------
# 5. BEFORE — random row-level 80/20 split
# ------------------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

random_metrics = evaluate_model(
    X_train_random,
    X_test_random,
    y_train_random,
    y_test_random
)

random_train_clients = set(
    feature_frame.loc[
        X_train_random.index,
        "client_hash_id"
    ]
)

random_test_clients = set(
    feature_frame.loc[
        X_test_random.index,
        "client_hash_id"
    ]
)

random_client_overlap = (
    random_train_clients.intersection(
        random_test_clients
    )
)


# ------------------------------------------------------------
# 6. AFTER — client-grouped 80/20 split
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx].copy()
X_test_grouped = X.iloc[test_idx].copy()

y_train_grouped = y.iloc[train_idx].copy()
y_test_grouped = y.iloc[test_idx].copy()

grouped_metrics = evaluate_model(
    X_train_grouped,
    X_test_grouped,
    y_train_grouped,
    y_test_grouped
)

grouped_train_clients = set(
    groups.iloc[train_idx]
)

grouped_test_clients = set(
    groups.iloc[test_idx]
)

grouped_client_overlap = (
    grouped_train_clients.intersection(
        grouped_test_clients
    )
)


# ------------------------------------------------------------
# 7. Before / after metrics
# ------------------------------------------------------------

comparison = pd.DataFrame(
    [
        random_metrics,
        grouped_metrics
    ],
    index=[
        "Before — Random row-level split",
        "After — Client-grouped split"
    ]
)

print("\nBEFORE / AFTER MODEL EVALUATION")
print("--------------------------------")

print(
    comparison.round(4).to_string()
)


# ------------------------------------------------------------
# 8. Split integrity checks
# ------------------------------------------------------------

print("\nBEFORE — RANDOM SPLIT")
print("---------------------")
print("Training rows:", len(X_train_random))
print("Test rows:", len(X_test_random))
print("Training clients:", len(random_train_clients))
print("Test clients:", len(random_test_clients))
print("Client overlap:", len(random_client_overlap))


print("\nAFTER — CLIENT-GROUPED SPLIT")
print("----------------------------")
print("Training rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))
print("Training clients:", len(grouped_train_clients))
print("Test clients:", len(grouped_test_clients))
print("Client overlap:", len(grouped_client_overlap))


Feature frame shape: (321546, 8)
Feature columns: ['feb_gsc_impressions', 'feb_gsc_clicks', 'feb_gsc_avg_position', 'feb_ga4_sessions', 'feb_scroll_events']
Positive label rate: 0.3176

BEFORE / AFTER MODEL EVALUATION
--------------------------------
                                 Accuracy  Precision  Recall      F1
Before — Random row-level split    0.8176     0.8933  0.4835  0.6274
After — Client-grouped split       0.7837     0.8896  0.4758  0.6200

BEFORE — RANDOM SPLIT
---------------------
Training rows: 257236
Test rows: 64310
Training clients: 54
Test clients: 54
Client overlap: 54

AFTER — CLIENT-GROUPED SPLIT
----------------------------
Training rows: 268596
Test rows: 52950
Training clients: 43
Test clients: 11
Client overlap: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

The model features are restricted to five signals aggregated from February 2026: GSC impressions, GSC clicks, GSC average position, GA4 sessions, and scroll events. The target is March activity and is kept separate from the feature set.

I checked the final feature names for references to the March outcome, label-derived information, and product or availability information. I also checked that every model feature is explicitly a February-derived feature.

The audit found no forbidden feature inputs. This supports the conclusion that the model inputs are temporally separated from the March outcome and do not directly encode the target.

This is a feature-level leakage audit. It does not prove that every possible source of bias has been eliminated.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================================

# Exact final feature set used by the Week-5 model
expected_features = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_ga4_sessions",
    "feb_scroll_events"
]

actual_features = list(feature_cols)

print("FINAL MODEL FEATURES")
print("--------------------")

for feature in actual_features:
    print("-", feature)


# ------------------------------------------------------------
# 1. Exact feature-set check
# ------------------------------------------------------------

feature_set_matches = (
    actual_features == expected_features
)


# ------------------------------------------------------------
# 2. Check for future / outcome-window references
# ------------------------------------------------------------

future_terms = [
    "march",
    "april",
    "may",
    "june",
    "future",
    "outcome"
]

future_feature_matches = [
    feature
    for feature in actual_features
    if any(term in feature.lower() for term in future_terms)
]


# ------------------------------------------------------------
# 3. Check for label-derived references
# ------------------------------------------------------------

label_terms = [
    "label",
    "target",
    "activity_label"
]

label_feature_matches = [
    feature
    for feature in actual_features
    if any(term in feature.lower() for term in label_terms)
]


# ------------------------------------------------------------
# 4. Check for product / availability / decision inputs
# ------------------------------------------------------------

forbidden_terms = [
    "product",
    "availability",
    "inventory",
    "stock",
    "decision",
    "action",
    "recommendation"
]

forbidden_feature_matches = [
    feature
    for feature in actual_features
    if any(term in feature.lower() for term in forbidden_terms)
]


# ------------------------------------------------------------
# 5. Check that the March target is not part of X
# ------------------------------------------------------------

target_in_features = (
    "march_activity_label" in X.columns
)


# ------------------------------------------------------------
# 6. Print audit results
# ------------------------------------------------------------

print("\nFEATURE SET CHECK")
print("-----------------")
print("Expected features match:", feature_set_matches)
print("Target in model features:", target_in_features)

print("\nFUTURE / OUTCOME CHECK")
print("----------------------")
print(
    "Forbidden future/outcome feature matches:",
    future_feature_matches
)

print("\nLABEL-DERIVED CHECK")
print("-------------------")
print(
    "Forbidden label feature matches:",
    label_feature_matches
)

print("\nPRODUCT / AVAILABILITY / DECISION CHECK")
print("---------------------------------------")
print(
    "Forbidden feature matches:",
    forbidden_feature_matches
)


# ------------------------------------------------------------
# 7. Final leakage result
# ------------------------------------------------------------

forbidden_inputs = (
    future_feature_matches
    + label_feature_matches
    + forbidden_feature_matches
)

leakage_check_passed = (
    feature_set_matches
    and not target_in_features
    and len(forbidden_inputs) == 0
)

print("\nFINAL LEAKAGE AUDIT")
print("-------------------")
print("Forbidden inputs found:", forbidden_inputs)
print("Leakage check passed:", leakage_check_passed)

FINAL MODEL FEATURES
--------------------
- feb_gsc_impressions
- feb_gsc_clicks
- feb_gsc_avg_position
- feb_ga4_sessions
- feb_scroll_events

FEATURE SET CHECK
-----------------
Expected features match: True
Target in model features: False

FUTURE / OUTCOME CHECK
----------------------
Forbidden future/outcome feature matches: []

LABEL-DERIVED CHECK
-------------------
Forbidden label feature matches: []

PRODUCT / AVAILABILITY / DECISION CHECK
---------------------------------------
Forbidden feature matches: []

FINAL LEAKAGE AUDIT
-------------------
Forbidden inputs found: []
Leakage check passed: True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim:**  
“February behavior alone is not sufficient to identify every future active item.”

**Safer claim:**  
“On the client-grouped test split, the Logistic Regression model measured 0.7837 accuracy, 0.8896 precision, 0.4758 recall, and 0.6200 F1. The observed false-negative rate was substantial, so the model provides directional decision-support from February signals but does not identify every March-active item.”

The safer version reports measured results from the unseen-client evaluation and avoids implying that the model can reliably identify all future active items. The result should be interpreted as directional decision-support rather than a guarantee of future activity.

### Real failure examples

I also inspect individual false positives and false negatives from the client-grouped test set. These examples show where the model's February signals did not match the observed March outcome.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 4 — CLAIM REWRITE + REAL FAILURE EXAMPLES
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 1. Re-train the same model on the honest grouped split
# ------------------------------------------------------------

audit_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(
        max_iter=1000,
        random_state=42
    )
)

audit_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_predictions = audit_model.predict(
    X_test_grouped
)


# ------------------------------------------------------------
# 2. Build an error table
# ------------------------------------------------------------

error_table = X_test_grouped.copy()

error_table["actual"] = y_test_grouped.values
error_table["predicted"] = grouped_predictions

error_table["error_type"] = np.select(
    [
        (error_table["actual"] == 1) &
        (error_table["predicted"] == 0),

        (error_table["actual"] == 0) &
        (error_table["predicted"] == 1)
    ],
    [
        "False negative",
        "False positive"
    ],
    default="Correct"
)


# ------------------------------------------------------------
# 3. Real false-negative examples
# ------------------------------------------------------------

false_negatives = error_table[
    error_table["error_type"] == "False negative"
].head(5)

print("REAL FALSE-NEGATIVE EXAMPLES")
print("----------------------------")
print(false_negatives.to_string(index=False))


# ------------------------------------------------------------
# 4. Real false-positive examples
# ------------------------------------------------------------

false_positives = error_table[
    error_table["error_type"] == "False positive"
].head(5)

print("\nREAL FALSE-POSITIVE EXAMPLES")
print("----------------------------")
print(false_positives.to_string(index=False))


# ------------------------------------------------------------
# 5. Error counts
# ------------------------------------------------------------

false_negative_count = (
    (error_table["actual"] == 1) &
    (error_table["predicted"] == 0)
).sum()

false_positive_count = (
    (error_table["actual"] == 0) &
    (error_table["predicted"] == 1)
).sum()

print("\nERROR COUNTS")
print("------------")
print("False negatives:", int(false_negative_count))
print("False positives:", int(false_positive_count))


# ------------------------------------------------------------
# 6. Safe interpretation
# ------------------------------------------------------------

print("\nSAFE CLAIM")
print("----------")
print(
    "The client-grouped evaluation provides measured, "
    "directional decision-support from February signals, "
    "but the observed false negatives show that it does "
    "not identify every March-active item."
)


REAL FALSE-NEGATIVE EXAMPLES
----------------------------
 feb_gsc_impressions  feb_gsc_clicks  feb_gsc_avg_position  feb_ga4_sessions  feb_scroll_events  actual  predicted     error_type
               273.0             0.0              4.925861               0.0                0.0       1          0 False negative
               279.0             0.0              8.024162               1.0                0.0       1          0 False negative
               746.0             0.0              7.767651               0.0                0.0       1          0 False negative
               933.0             0.0             33.847691               0.0                0.0       1          0 False negative
               485.0             0.0             32.250178               0.0                0.0       1          0 False negative

REAL FALSE-POSITIVE EXAMPLES
----------------------------
 feb_gsc_impressions  feb_gsc_clicks  feb_gsc_avg_position  feb_ga4_sessions  feb_scroll_events  actual

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.